# 08.11 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/11-pytorch/exercises.ipynb)

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
if torch.cuda.is_available():
    device = "cuda"                     # NVIDIA GPU (Windows / Linux)
elif torch.backends.mps.is_available():
    device = "mps"                      # Apple Silicon GPU
else:
    device = "cpu"
print("PyTorch", torch.__version__, "· running on", device)

from pathlib import Path

def models_dir():
    """The repo-root models/ folder (git-ignored); created if missing."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "data").is_dir() and (parent / "modules").is_dir():
            folder = parent / "models"
            break
    else:
        folder = Path("models")
    folder.mkdir(exist_ok=True)
    return folder

from torchvision import datasets

def data_dir():
    """data/raw/ in the repo (git-ignored), or a local folder on Colab."""
    return models_dir().parent / "data" / "raw"

train_raw = datasets.FashionMNIST(data_dir(), train=True, download=True)
test_raw = datasets.FashionMNIST(data_dir(), train=False, download=True)
CLASSES = train_raw.classes
print(len(train_raw), "training photos,", len(test_raw), "test photos")

class ProductPhotos(Dataset):
    """Hands out (photo, label) pairs one at a time; DataLoader groups them into batches."""
    def __init__(self, raw):
        self.x = raw.data.float().div(255).unsqueeze(1)   # (N, 1, 28, 28): channels come FIRST in PyTorch
        self.y = raw.targets
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_ds, test_ds = ProductPhotos(train_raw), ProductPhotos(test_raw)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=512)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 5 * 5, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x):
        return self.head(self.features(x))          # raw scores ("logits"), no softmax

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()                                    # dropout etc. switch on
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()                        # 1. forget last batch's gradients
        loss = loss_fn(model(xb), yb)                # 2. forward pass and loss
        loss.backward()                              # 3. backpropagation (08.4)
        optimizer.step()                             # 4. gradient-descent update
        total += loss.item() * len(yb)
    return total / len(loader.dataset)

@torch.no_grad()
def accuracy(model, loader):
    model.eval()                                     # dropout etc. switch off
    correct = 0
    for xb, yb in loader:
        correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return correct / len(loader.dataset)

## Exercise 1: count the parameters by hand

Count the parameters of `SmallCNN` layer by layer (`Conv2d(1, 16, 3)`, `Conv2d(16, 32, 3)`, `Linear(800, 64)`, `Linear(64, 10)`), then check with `n_params`.

In [ ]:
# your code here

## Exercise 2: add dropout

Add `nn.Dropout(0.3)` after `nn.Flatten()` in a copy of the class. Train it for 3 epochs. Why must you call `model.eval()` before measuring accuracy?

In [ ]:
# your code here

## Exercise 3: a confusion matrix

Write a function that returns the true and predicted labels for the whole test set, then print the five most common mistakes with `sklearn.metrics.confusion_matrix`. Are they the same as in 08.8?

In [ ]:
# your code here

## Exercise 4: save and load

Train a model for one epoch, save its state dict, load it into a fresh `SmallCNN()` and check that both give exactly the same predictions on the test set.

In [ ]:
# your code here